# Methodological Framework

__TABLE OF CONTENTS__

1. [Cross-Validation](#cv)  
    1.1. [K-fold configuration](#k-fold)  
    1.2. [Shared evaluation metrics](#eval)
2. [Preprocessing Structure](#preproc)
3. [Model Assessment and Ablation Study](#ablation)
4. [Closing Summary + Github Link](#summary)

<a id="cv"></a>
## 1. Cross-Validation

To obtain a reliable estimate of out-of-sample performance and to compare models under the same conditions, all linear and regularized models in this notebook are evaluated using the same cross-validation framework. This avoids overfitting to a single train/validation split and ensures that performance comparisons between models are fair and reproducible.

<a id="k-fold"></a>
### 1.1. K-Fold configuration

We use an **8-fold K-Fold** cross-validation scheme:
- The data is split into `N_SPLITS = 8` approximately equal folds.
- At each iteration, 1 fold is used as the **validation set** and the remaining 7 folds are used as the **training set**.
- `shuffle = True` is used to randomize the split, and a fixed `RANDOM_STATE = 42` guarantees reproducibility of the folds.
- This configuration is applied consistently across all models (Linear, Ridge, Lasso, Elastic Net), so differences in performance reflect the models themselves rather than differences in the data split.

For each model and configuration, we record per-fold metrics for both **train** and **validation** sets and later aggregate them to summarize performance.

<a id="eval"></a>
### 1.2. Shared evaluation metrics

All models are observed with the same set of regression metrics:

- **RMSE (Root Mean Squared Error):** Penalizes larger errors more strongly; used as the primary measure of overall prediction error. 

  $$
  \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
  $$


- **MAE (Mean Absolute Error):** Measures the average absolute deviation between predictions and true values; more robust to outliers than RMSE; the one considered in Kaggle results. 

  $$
  \mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
  $$


- **\(R^2\) (Coefficient of Determination):** Indicates the proportion of variance in the target explained by the model. 

  $$
  R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
  $$


- **Bias (Signed Error):**  Captures systematic overprediction (positive bias) or underprediction (negative bias).

  $$
  \text{Bias} = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)
  $$


For each fold we compute these metrics on both **train** and **validation** sets. Comparing train vs validation values allows us to diagnose underfitting/overfitting, while comparing across models (using mean validation metrics over folds) provides a consistent basis for model selection.

Even though we observe all these metrics for every model, **our goal is to minimize the RMSE**, and if there are two models with very similar RMSEs we then use the MAE as a tie-breaker.  

__We prioritize RMSE because, in car pricing, one massive mistake (e.g., valuing a €50k car at €30k) is much worse than many small mistakes__. RMSE gives much more weight to large errors than MAE, so selecting models based on RMSE favors solutions that better control extreme mispricing. We then use MAE as a secondary metric to assess and refine performance on typical “average-case” predictions.

<a id="preproc"></a>
## 2. Preprocessing Structure

### 2.1. Standardization and Data Correction

All models share a common, leakage-safe preprocessing structure to ensure fair comparison and reproducibility.

- **Deterministic cleaning:**  
    - As discussed in the EDA, we exclude identifiers (carID), mechanic-dependent features (paintQuality%) and constant variables (hasDamage). Invalid/missing categorical values are encoded as UNKNOWN.
    - We apply preprocess_categorical to avoid inconsistent category strings by (i) replacing NaNs with UNKNOWN (fill_unknown) and (ii) normalizing text with basic_string_transformer: trim spaces, uppercase, remove accents, remove internal spaces and strip non-alphanumeric characters (keeping only allowed extras). This is leakage-safe because it does not use training statistics.
    - The tax preprocessing function is also deterministic.

- **Fit and Transform Logic:**  
    - The remaining cases follow a standard fit/transform workflow: fit is performed using only the training fold (e.g., to compute medians or other statistics) and transform is then applied to both the training and validation/test splits.

The output of preprocessing is a fully aligned feature matrix with identical columns across train, validation, and test sets, enabling consistent evaluation and deployment.

### 2.2. Encoding

Categorical encoding was applied to four variables: model, Brand, fuelType, and transmission. Based on the cardinality of each variable, two different encoding techniques were strategically chosen: Target Encoding and One-Hot Encoding.

#### 2.2.1 One-Hot Encoding

The variables fuelType and transmission were encoded using One-Hot Encoding, as they exhibit a limited, well-defined set of categories. In such cases, one-hot encoding is appropriate because it clearly separates categories without creating an excessive number of additional features.

During training, the encoder learns the set of categories present in the data. At transformation time, these categories are converted into binary indicator (0/1) columns, while any categories not observed during training are safely ignored, ensuring a consistent feature space across training and validation data.

#### 2.2.2. Target Encoding

In contrast, Brand and model were encoded using Target Encoding due to their large number of distinct categories. Applying one-hot encoding to these variables would have resulted in a very high-dimensional and sparse feature space, significantly increasing computational cost and the risk of overfitting. Target encoding addresses this issue by replacing each category with a statistic derived from the target variable, specifically, the mean target value associated with that category, which avoids any increase in the number of feature columns.

##### 2.2.2.1. Target Encoding: Data Leakage Considerations

However, if not implemented carefully, target encoding can lead to data leakage, since the encoding depends directly on target information. To prevent this, the encoder is fitted exclusively on the training data within each cross-validation fold and then applied to the corresponding validation data. Categories that are not observed during the fit stage are not learned and are instead mapped to a default value (media global) during transformation.

##### 2.2.2.2. Target Encoding: Overfitting and Smoothing

Additionally, target encoding may still introduce overfitting, particularly for rare categories whose target means are estimated from very few samples. In our MyTargetEncoder, this risk is mitigated through a smoothing mechanism that blends the category-specific mean with the global target mean based on the number of observations per category. This ensures that unreliable estimates from rare categories are pulled toward a more stable global value.

Unlike standard target encoders that use linear smoothing, our implementation employs a sigmoid-based weighting function, which keeps low-frequency categories close to the global target mean until a predefined sample threshold is reached, effectively preventing category means from having a significant influence before they become statistically reliable. Once this threshold is exceeded, the encoder behaves similarly to standard approaches, progressively assigning more weight to the category mean as the number of observations increases, resulting in stable encodings and reduced overfitting caused by rare-category outliers.

### 2.3. Scaling 

We apply *StandardScaler()* to all numeric features to place them on a comparable scale (zero mean, unit variance), preventing variables measured in large units (e.g., mileage, tax) from disproportionately influencing distance-based optimization. This is especially important for models that rely on Euclidean geometry or penalization, such as linear models with L1/L2 regularization, where unscaled inputs can distort neighborhoods and shrinkage. For tree-based learners scaling is typically unnecessary, but keeping the same scaling step across all experiments ensures a consistent preprocessing interface and fair model comparisons; in every CV fold, the scaler is fit only on the training split and then applied to the validation split to avoid leakage.

### 2.4. Feature Selection (FS)

- Our feature selection strategy aimed to improve the predictive performance of the strongest models at that stage: __Random Forest Regressor__ and __Histogram Gradient Boosting__. Since Histogram Gradient Boosting achieved the best baseline results without feature selection, we performed the feature selection search primarily on this model, while ensuring the approach remained compatible with Random Forest (and the rest of the models).

- Given this objective, we adopted an __embedded feature selection__ approach using a __Random Forest estimator inside SelectFromModel__.

- We searched for the best proportion of top ranked features to retain. We initially experimented with the threshold parameter, but results were consistently poor and not informative, so we excluded those experiments from the final version. The more exhaustive search was conducted only for Histogram Gradient Boosting, and the best performing settings are documented in the corresponding notebook.

- We ran two separate studies:
    - (i) Original + engineered feature set (excluding columns removed in the EDA stage).
    - (ii) Same setting, but excluding previousOwners and including the engineered age feature.

- For (i), we tested retention ratios of 40%, 55%, 60%, 65%, 70%, 75%, and 80%. We found that 65% (17 features) and 70% (18 features) performed best overall, with 65% yielding the strongest results. In contrast, 40% (10 features) was overly restrictive and produced the worst performance among the tested ratios.

- For (ii), the feature space was smaller (15 features before selection), so we did not conduct an exhaustive search. Retaining 90% produced no practical change, so we selected 80% of the most relevant features. However, for Histogram Gradient Boosting, performance with feature selection was slightly worse, although the differences were negligible overall.

- Overall, we used:
    - 65% feature retention for experiments combining original and engineered features (as the best-performing setting in study (i));
    - for the reduced set (excluding previousOwners and including age), we reported results for both 80% retention and no feature selection, since the observed differences were minimal and could interact differently with other model families.

### 2.5. Feature Engineering (FE)

We engineered a small set of features to better capture non-linear depreciation and usage effects. In practice, these transformations replace variables with more informative representations (e.g., age instead of year, log-scaled mileage, usage intensity per year) and introduce compact categorical signals (engine-size bins and ownership flags) that are easier for the models to exploit.

- __create_age_and_drop_year__: creates age = 2020 − year (clipping negative ages to 0, although just for safety since they were already handled) and drops the original year column.

- __add_owners_flagged__: creates a binary feature owners_flagged (1 if previousOwners > 1, else 0) and can drop previousOwners.

- __add_mileage_features__: creates log_mileage = log(1 + mileage) and log_miles_per_year = log(1 + mileage / max(age,1)), then drops miles_per_year and/or the original mileage depending on the flags.

- __add_engine_bins__: discretizes engineSize into an ordinal categorical feature engine_bin using predefined cut points (e.g., 0-1.0, 1.0-1.3, …, 4.0+), keeping the original engineSize.

<a id="ablation"></a>
## 3. Model Assessment and Ablation Study

To quantify the impact of specific design choices (without confounding from different CV splits or preprocessing), we benchmark **five standardized variants** across model families under the same **8-fold cross-validation** setup. In every fold, **all preprocessing, feature engineering, and feature selection steps are fit on the training split only** and then applied to the validation split (leakage-safe). Hyperparameters are explored via a **reduced random search**, reusing the same sampled configurations across folds for fair comparison. When using a log target, models are trained on `log1p(price)`, but predictions are mapped back with `expm1()` before computing RMSE/MAE/R2/bias (all reported back to price).

**The five variants are:**
1. **Baseline (raw target):** original features, no feature engineering (FE), no feature selection (FS), target = `price`.
2. **No `previousOwners` + log target:** drop `previousOwners`, target = `log1p(price)`.
3. **Age instead of year:** variant (2) + replace `year` with `age` (to represent depreciation more directly).
4. **Feature selection (80%):** variant (3) + keep the **top 80%** of features using a model-based selector (RF + `SelectFromModel`) fitted within each training fold.
5. **Full FE + FS (65%):** variant (3) + add engineered features (`owners_flagged`, mileage-based features, `engine_bin`) and retain the **top 65%** of features via the same CV-safe FS procedure.

For transparency and reproducibility, we export per-variant search results and build a compact leaderboard reporting the **best configuration per variant** (ranked by validation RMSE).

### 3.1. Rationale Behind Excluding previousOwners

- We decided to conduct this ablation study because, during the EDA step, we noticed some unusual behaviour in the previousOwners column. For example, we observed multiple cars that were at most 5 years old with 4 previous owners, and we also found that the year with the highest number of cars reporting 6 previous owners was 2019 (i.e., cars that were only 1 year old). The plots supporting these observations are included in the EDA notebook.

- During preprocessing, we did not attempt to correct these values because doing so could introduce additional errors. Even if the reported number of previous owners is incorrect (e.g., 4 or 6), we have no reliable way to infer the true value (e.g., whether it should be 1, 2, or 3). Moreover, there may be valid explanations for these atypical cases. Therefore, we considered leaving the feature unchanged (and evaluating its impact via ablation) to be the most appropriate approach.

### 3.2. Target Transformation: Log(price)

- In our EDA, we observed that the price target is right-skewed with a relatively heavy upper tail: most cars cluster at lower prices, while a smaller number of high priced vehicles extend the distribution toward much larger values. In this setting, squared-error based objectives (e.g., MSE/RMSE) can overweight a few expensive cars and prediction errors often become __heteroscedastic__, with larger variance at higher price levels.

- We log-transform the target to make errors more stable across the price range (reducing heteroscedasticity), to limit the influence of extreme high-price cars by compressing the upper tail and to better reflect relative (percentage-like) errors. We still report RMSE on the original price scale by applying the inverse transform to predictions.

<a id="summary"></a>
## 4. Closing Summary

This notebook formalizes the methodological backbone used throughout the project to ensure that model comparisons are **fair, reproducible, and leakage-free**. We adopt a fixed **8-fold** configuration and report a shared set of regression metrics across all experiments, using **RMSE as the primary objective** (with MAE and the remaining metrics used for complementary diagnostics).

To avoid over-optimistic estimates (data-leakage), all preprocessing steps are applied under a strict **fit-on-train / transform-on-validation** rule within each fold. This includes deterministic cleaning, missing-value handling, and model-dependent encoding/scaling — always producing an aligned feature matrix across train/validation/test.

Finally, we run a controlled ablation study based on **five standardized variants** (target scaling, feature removal, age representation, and FE/FS), combined with reduced random searches to balance performance exploration and computational cost. This framework provides a consistent basis to interpret performance changes as the result of explicit modeling decisions, and it supports reliable selection of the final approach to minimize generalization error.

Our whole project is available on [INSERT LINK].